# 🎓 Detección por Plantillas (Template Matching)

A veces no buscamos un "círculo" o un "color", sino un objeto exacto (un logo, una cara, un componente). Aquí aprenderemos a buscar una pequeña imagen dentro de otra más grande.

### 🎯 Objetivos de Aprendizaje
1.  Entender la técnica de **Match Template**.
2.  Configurar umbrales de coincidencia (Thresholding).
3.  Localizar múltiples objetos idénticos.

---

## 1. ¿Cómo funciona?

OpenCV desliza la "Plantilla" sobre la imagen principal y calcula una puntuación de similitud en cada píxel.

In [ ]:
import cv2
import numpy as np

# 1. Cargar imagen principal y plantilla
img_rgb = cv2.imread('estanteria_productos.jpg')
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2GRAY)
template = cv2.imread('logo_producto.jpg', 0)
w, h = template.shape[::-1] # Guardar dimensiones para dibujar el cuadro

# 2. Ejecutar Matching
res = cv2.matchTemplate(img_gray, template, cv2.TM_CCOEFF_NORMED)

# 3. Encontrar la mejor coincidencia
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(res)

# max_loc es la esquina superior izquierda del mejor match
top_left = max_loc
bottom_right = (top_left[0] + w, top_left[1] + h)

cv2.rectangle(img_rgb, top_left, bottom_right, (0, 255, 0), 2)
cv2.imshow('Detección', img_rgb)
cv2.waitKey(0)

### 🚀 Proyecto Aplicado: "Inspector de Placas de Circuito"

En una fábrica de electrónica, debemos verificar si un condensador específico está presente en la placa. Vamos a buscar múltiples coincidencias.

In [ ]:
# =================================================================
# PROYECTO: Inspector de Calidad de Placas de Circuito (PCBs)
# OBJETIVO: Localizar múltiples componentes (condensadores) usando Template Matching
# =================================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------------------------------------------
# 1. GENERACIÓN DE DATOS SINTÉTICOS (Solo para que el código funcione de inmediato)
# -----------------------------------------------------------------
def crear_datos_prueba():
    # Crear una "Placa" (Escena)
    placa = np.full((600, 800, 3), (200, 200, 200), dtype=np.uint8)
    # Dibujar componentes genéricos
    for i in range(3):
        for j in range(3):
            x, y = 100 + i*250, 100 + j*180
            # Caja del componente (Azul)
            cv2.rectangle(placa, (x, y), (x+80, y+120), (150, 100, 50), -1)
            # El Logo/Marca que buscaremos (Círculo rojo con X)
            cv2.circle(placa, (x+40, y+40), 15, (0, 0, 255), -1)
            cv2.line(placa, (x+35, y+35), (x+45, y+45), (255, 255, 255), 2)
            cv2.line(placa, (x+45, y+35), (x+35, y+45), (255, 255, 255), 2)
    
    # Crear la "Plantilla" (El componente que buscamos)
    plantilla = placa[100:220, 100:180].copy()
    
    cv2.imwrite('placa_circuito.jpg', placa)
    cv2.imwrite('condensador_modelo.jpg', plantilla)
    print("✅ Imágenes 'placa_circuito.jpg' y 'condensador_modelo.jpg' creadas.")

# Ejecutamos la creación de imágenes
crear_datos_prueba()

# -----------------------------------------------------------------
# 2. PROCESAMIENTO DE VISIÓN ARTIFICIAL
# -----------------------------------------------------------------

# Paso 1: Cargar imágenes
img_rgb = cv2.imread('placa_circuito.jpg')
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2GRAY)
template = cv2.imread('condensador_modelo.jpg', 0)
w, h = template.shape[::-1] # Obtener dimensiones del modelo

# Paso 2: Ejecutar Template Matching
# Usamos TM_CCOEFF_NORMED para obtener un valor de 0 a 1 (donde 1 es perfecto)
res = cv2.matchTemplate(img_gray, template, cv2.TM_CCOEFF_NORMED)

# Paso 3: Filtrar por Umbral (Threshold)
# Queremos coincidencias mayores al 80%
threshold = 0.8
loc = np.where(res >= threshold)

# Paso 4: Marcar resultados
img_final = img_rgb.copy()
puntos_encontrados = 0

# 'loc' devuelve (y, x). Los invertimos para dibujar (x, y)
for pt in zip(*loc[::-1]):
    # Dibujar el rectángulo del hallazgo
    cv2.rectangle(img_final, pt, (pt[0] + w, pt[1] + h), (0, 255, 0), 2)
    puntos_encontrados += 1

# Nota: El conteo de 'puntos_encontrados' aquí puede ser alto porque 
# detecta píxeles vecinos. En la vida real se usa 'Non-Maximum Suppression'.

# -----------------------------------------------------------------
# 3. VISUALIZACIÓN PEDAGÓGICA
# -----------------------------------------------------------------

plt.figure(figsize=(15, 10))

# Mostrar la plantilla que buscábamos
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(cv2.imread('condensador_modelo.jpg'), cv2.COLOR_BGR2RGB))
plt.title("Modelo a buscar (Template)")
plt.axis('off')

# Mostrar el resultado en la placa
plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(img_final, cv2.COLOR_BGR2RGB))
plt.title(f"Resultado de Inspección: Componentes detectados")
plt.axis('off')

plt.tight_layout()
plt.show()

print(f"--- REPORTE DE CALIDAD ---")
print(f"Estado: Procesado")
print(f"Hallazgos: Se localizaron las zonas que coinciden con el modelo.")

## Registro de Etiquetas

In [ ]:
import cv2
import numpy as np

# 1. Configuración inicial
cap = cv2.VideoCapture(2)
GUI_SIZE = 150 # Tamaño del cuadro amarillo para capturar

# Diccionario para guardar nuestras etiquetas/plantillas
# Guardaremos: 'nombre': [imagen_gris, (ancho_orig, alto_orig)]
templates = {}

def buscar_multiescala(frame_gray, tpl, threshold=0.7):
    """Busca una plantilla en diferentes escalas dentro de una imagen."""
    tH, tW = tpl.shape[:2]
    found = None

    # Probamos redimensionar la imagen de la cámara del 100% al 20%
    for scale in np.linspace(0.2, 1.0, 15)[::-1]:
        resized = cv2.resize(frame_gray, (int(frame_gray.shape[1] * scale), int(frame_gray.shape[0] * scale)))
        ratio = frame_gray.shape[1] / float(resized.shape[1])

        # Si la imagen se vuelve más pequeña que la plantilla, paramos
        if resized.shape[0] < tH or resized.shape[1] < tW:
            break

        # MatchTemplate en esta escala
        res = cv2.matchTemplate(resized, tpl, cv2.TM_CCOEFF_NORMED)
        (_, maxVal, _, maxLoc) = cv2.minMaxLoc(res)

        # Guardamos la mejor coincidencia de todas las escalas
        if found is None or maxVal > found[0]:
            found = (maxVal, maxLoc, ratio)

    if found and found[0] >= threshold:
        return found
    return None

print("INSTRUCCIONES:")
print("1. Pon el logo en el cuadro amarillo.")
print("2. Presiona 'c' para capturar Coca-Cola o 'p' para Pepsi.")
print("3. Una vez capturado, muévelo a cualquier distancia.")

while True:
    ret, frame = cap.read()
    if not ret: break
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    h_f, w_f = gray.shape
    
    # Dibujar cuadro guía central
    x1, y1 = (w_f // 2) - (GUI_SIZE // 2), (h_f // 2) - (GUI_SIZE // 2)
    x2, y2 = x1 + GUI_SIZE, y1 + GUI_SIZE
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 2)

    # --- PROCESAR DETECCIONES ---
    for nombre, (tpl, dims) in templates.items():
        deteccion = buscar_multiescala(gray, tpl)
        
        if deteccion:
            maxVal, maxLoc, ratio = deteccion
            tW, tH = dims
            # Escalar coordenadas de vuelta al tamaño original del video
            startX, startY = (int(maxLoc[0] * ratio), int(maxLoc[1] * ratio))
            endX, endY = (int((maxLoc[0] + tW) * ratio), int((maxLoc[1] + tH) * ratio))

            color = (0, 255, 0) if nombre == "COKE" else (255, 0, 0)
            cv2.rectangle(frame, (startX, startY), (endX, endY), color, 2)
            cv2.putText(frame, f"{nombre} {int(maxVal*100)}%", (startX, startY-10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    cv2.imshow('Detector de Stock Multiescala', frame)

    # --- CAPTURA DE ETIQUETAS ---
    key = cv2.waitKey(1) & 0xFF
    if key == ord('c'):
        roi = gray[y1:y2, x1:x2]
        # Guardamos la plantilla y sus dimensiones
        templates["COKE"] = [roi, (roi.shape[1], roi.shape[0])]
        print("✅ Coca-Cola registrada.")
    elif key == ord('p'):
        roi = gray[y1:y2, x1:x2]
        templates["PEPSI"] = [roi, (roi.shape[1], roi.shape[0])]
        print("✅ Pepsi registrada.")
    elif key == ord('q'): break

cap.release()
cv2.destroyAllWindows()

## Sistema de Inventario Inteligente con Registro y Voz

In [1]:
import cv2
import numpy as np
import time
import threading
import queue
import platform

# ============================
#  TTS ROBUSTO (NO BLOQUEA)
# ============================
import pyttsx3

tts_queue = queue.Queue()

def tts_worker():
    """
    Hilo dedicado SOLO a hablar.
    Así OpenCV nunca se congela y el audio no se pierde.
    """
    # En Windows conviene forzar el driver sapi5
    if platform.system().lower().startswith("win"):
        engine = pyttsx3.init(driverName="sapi5")
    else:
        engine = pyttsx3.init()

    engine.setProperty("rate", 180)

    # (Opcional) elegir una voz disponible
    # voices = engine.getProperty("voices")
    # if voices:
    #     engine.setProperty("voice", voices[0].id)

    while True:
        texto = tts_queue.get()
        if texto is None:
            break
        try:
            engine.say(texto)
            engine.runAndWait()
        except Exception as e:
            # Si falla el audio, no matamos el programa, pero lo reportamos
            print("⚠️ Error TTS:", e)

def hablar(texto):
    """
    En vez de hablar aquí (bloquear), lo mandamos a la cola.
    """
    print(f"Asistente: {texto}")
    tts_queue.put(texto)

# Arrancamos el hilo de voz (daemon para que no impida cerrar)
threading.Thread(target=tts_worker, daemon=True).start()


# ============================
#  BEEP REAL (WINDOWS)
# ============================
def beep():
    if platform.system().lower().startswith("win"):
        try:
            import winsound
            winsound.Beep(1200, 120)  # frecuencia, duración(ms)
        except:
            pass
    else:
        # En otros SO, intentamos el beep simple
        print("\a", end="")


# ============================
#  FUNCIONES DE APOYO
# ============================
def buscar_multiescala(frame_gray, tpl, threshold=0.7):
    tH, tW = tpl.shape[:2]
    found = None

    for scale in np.linspace(0.2, 1.2, 15)[::-1]:
        resized = cv2.resize(
            frame_gray,
            (int(frame_gray.shape[1] * scale), int(frame_gray.shape[0] * scale))
        )
        ratio = frame_gray.shape[1] / float(resized.shape[1])

        if resized.shape[0] < tH or resized.shape[1] < tW:
            break

        res = cv2.matchTemplate(resized, tpl, cv2.TM_CCOEFF_NORMED)
        (_, maxVal, _, maxLoc) = cv2.minMaxLoc(res)

        if found is None or maxVal > found[0]:
            found = (maxVal, maxLoc, ratio)

    if found and found[0] >= threshold:
        return found
    return None


# ============================
#  VARIABLES GLOBALES
# ============================
cap = cv2.VideoCapture(0)
GUI_SIZE = 180
base_datos = {}
ultima_vez_hablado = {}


# ==========================================
# FASE 1: REGISTRO DE PRODUCTOS
# ==========================================
print("--- INICIANDO MÓDULO DE REGISTRO ---")
hablar("Iniciando registro. Coloque el producto en el cuadro y presione la tecla C")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape
    x1, y1 = (w//2)-(GUI_SIZE//2), (h//2)-(GUI_SIZE//2)
    x2, y2 = x1+GUI_SIZE, y1+GUI_SIZE

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 2)
    cv2.putText(frame, "REGISTRO: Pulsa 'C' para capturar", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)

    cv2.imshow('Supermercado IA', frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('c'):
        beep()

        # NOTA: input() pausa el video (normal), pero no afecta el audio ya
        nombre = input("Nombre del producto: ")
        s_min = int(input(f"Stock mínimo para {nombre}: "))

        roi_gray = cv2.cvtColor(frame[y1:y2, x1:x2], cv2.COLOR_BGR2GRAY)
        base_datos[nombre] = [roi_gray, (roi_gray.shape[1], roi_gray.shape[0]), s_min]

        hablar(f"{nombre} registrado")

        opcion = input("¿Deseas registrar otro? (s/n): ")
        if opcion.lower() == 'n':
            break

# ==========================================
# FASE 2: MONITOREO (PANTALLA COMPLETA)
# ==========================================
hablar("Iniciando escáner. Buscando productos registrados.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    inventario_actual = {nombre: 0 for nombre in base_datos.keys()}

    for nombre, info in base_datos.items():
        plantilla, dims, s_min = info
        det = buscar_multiescala(gray, plantilla)

        if det:
            maxVal, maxLoc, ratio = det
            startX = int(maxLoc[0] * ratio)
            startY = int(maxLoc[1] * ratio)

            cv2.rectangle(frame, (startX, startY), (startX+dims[0], startY+dims[1]), (0, 255, 0), 2)
            cv2.putText(frame, f"OK: {nombre}", (startX, startY-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

            inventario_actual[nombre] += 1

            

    # Mostrar Inventario
    y_off = 60
    for n, cant in inventario_actual.items():
        color = (0, 255, 0) if cant >= base_datos[n][2] else (0, 0, 255)
        cv2.putText(frame, f"{n}: {cant}/{base_datos[n][2]}",
                    (20, y_off), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)
        y_off += 35

    cv2.imshow('Supermercado IA', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# Apagamos el hilo TTS limpiamente
tts_queue.put(None)


--- INICIANDO MÓDULO DE REGISTRO ---
Asistente: Iniciando registro. Coloque el producto en el cuadro y presione la tecla C
Asistente: Topiramato registrado
Asistente: Iniciando escáner. Buscando productos registrados.


In [5]:
import cv2
import numpy as np
import os
import json
import time
import random
from dataclasses import dataclass

import tkinter as tk
from tkinter import simpledialog

# ============================================================
# CONFIGURACIÓN
# ============================================================
CFG = {
    "CAM_INDEX": 0,
    "ROI_SIZE": 180,          # tamaño del recuadro central para capturar template
    "SCALES": 15,             # escalas para matchTemplate
    "SCALE_MIN": 0.30,
    "SCALE_MAX": 1.00,
    "THRESHOLD": 0.78,        # umbral matchTemplate
    "TRAIN_SPLIT": 0.80,      # train/val
    "SAVE_JPEG_QUALITY": 95,
    "SHOW_DEBUG": True,
    "AUTO_SAVE_EVERY": 10,
}

# ============================================================
# UTILIDADES
# ============================================================
def safe_name(s: str) -> str:
    return s.strip().replace(" ", "_")

def ensure_dirs(base):
    paths = [
        f"{base}/images/train", f"{base}/images/val",
        f"{base}/labels/train", f"{base}/labels/val",
        f"{base}/images/background/train", f"{base}/images/background/val",
        f"{base}/templates"
    ]
    for p in paths:
        os.makedirs(p, exist_ok=True)

def write_data_yaml(project_dir, classes):
    names = [None] * len(classes)
    for name, cid in classes.items():
        names[cid] = name

    yaml_content = (
        "train: images/train\n"
        "val: images/val\n\n"
        f"nc: {len(names)}\n"
        f"names: {names}\n"
    )
    with open(os.path.join(project_dir, "data.yaml"), "w", encoding="utf-8") as f:
        f.write(yaml_content)

def save_project_meta(project_dir, meta):
    with open(os.path.join(project_dir, "project.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

def yolo_label_line(class_id, bx, by, bw, bh, img_w, img_h):
    x_center = (bx + bw / 2) / img_w
    y_center = (by + bh / 2) / img_h
    nw = bw / img_w
    nh = bh / img_h

    # clamp
    x_center = min(max(x_center, 0.0), 1.0)
    y_center = min(max(y_center, 0.0), 1.0)
    nw = min(max(nw, 0.0), 1.0)
    nh = min(max(nh, 0.0), 1.0)

    return f"{class_id} {x_center:.6f} {y_center:.6f} {nw:.6f} {nh:.6f}"

def get_class_name(classes_dict, class_id):
    for name, cid in classes_dict.items():
        if cid == class_id:
            return name
    return "N/A"

def ask_text_window(title, prompt):
    """
    Ventana para pedir texto (ideal en Jupyter).
    Retorna None si cancelan.
    """
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    value = simpledialog.askstring(title, prompt)
    root.destroy()
    return value

# ============================================================
# MATCH TEMPLATE MULTIESCALA
# ============================================================
@dataclass
class Detection:
    score: float
    bx: int
    by: int
    bw: int
    bh: int

def multiscale_match(frame_gray, tpl_gray, threshold=0.78):
    tH, tW = tpl_gray.shape[:2]
    best = None

    scales = np.linspace(CFG["SCALE_MIN"], CFG["SCALE_MAX"], CFG["SCALES"])[::-1]
    for scale in scales:
        resized = cv2.resize(
            frame_gray,
            (int(frame_gray.shape[1] * scale), int(frame_gray.shape[0] * scale))
        )
        if resized.shape[0] < tH or resized.shape[1] < tW:
            break

        res = cv2.matchTemplate(resized, tpl_gray, cv2.TM_CCOEFF_NORMED)
        _, maxVal, _, maxLoc = cv2.minMaxLoc(res)

        ratio = frame_gray.shape[1] / float(resized.shape[1])
        if best is None or maxVal > best[0]:
            best = (maxVal, maxLoc, ratio)

    if best and best[0] >= threshold:
        score, (x, y), ratio = best
        bx = int(x * ratio)
        by = int(y * ratio)
        bw = int(tW * ratio)
        bh = int(tH * ratio)
        return Detection(score=score, bx=bx, by=by, bw=bw, bh=bh)

    return None

# ============================================================
# CAPTURA TEMPLATE
# ============================================================
def capture_template(cap, class_name):
    """
    Captura un template (ROI central). Presiona 'c' para guardar, 'q' para cancelar.
    """
    roi = CFG["ROI_SIZE"]
    print(f"\n--- Captura de TEMPLATE para: {class_name} ---")
    print("Coloca el objeto centrado y presiona 'c' para capturar (q para cancelar).")
    print("Tip: da CLICK en la ventana 'Captura Template' antes de presionar teclas.")

    while True:
        ret, frame = cap.read()
        if not ret:
            return None

        h, w = frame.shape[:2]
        x1, y1 = (w // 2) - (roi // 2), (h // 2) - (roi // 2)
        x2, y2 = x1 + roi, y1 + roi

        vis = frame.copy()
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 255), 2)
        cv2.putText(
            vis,
            f"TEMPLATE: {class_name} (c=guardar, q=cancelar)",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 255),
            2
        )

        cv2.imshow("Captura Template", vis)
        key = cv2.waitKey(1) & 0xFF

        if key == ord('c'):
            tpl = frame[y1:y2, x1:x2]
            tpl_gray = cv2.cvtColor(tpl, cv2.COLOR_BGR2GRAY)
            cv2.destroyWindow("Captura Template")
            return tpl_gray

        if key == ord('q'):
            cv2.destroyWindow("Captura Template")
            return None

# ============================================================
# FLUJO PRINCIPAL
# ============================================================
def main():
    print("=== GENERADOR DE DATASET (YOLO)  ===")
    project = safe_name(input("Nombre del proyecto (ej. Frutas_YOLO): ").strip())
    if not project:
        print("Proyecto inválido.")
        return

    split_ratio_str = input(f"Porcentaje train (0.0 a 1.0) [default {CFG['TRAIN_SPLIT']}]: ").strip()
    if split_ratio_str:
        CFG["TRAIN_SPLIT"] = float(split_ratio_str)

    project_dir = project
    ensure_dirs(project_dir)

    classes = {}     # name -> id
    templates = {}   # id -> tpl_gray
    counters = {
        "images_total": 0,
        "per_class": {},
        "background_total": 0
    }

    def add_class_flow(cap):
        """
        En Jupyter: pregunta nombre con ventana, NO con input().
        """
        raw = ask_text_window("Nueva clase", "Nombre del nuevo objeto/clase (ej. manzana):")
        if raw is None:
            print("Cancelado: no se agregó clase.")
            return None

        name = safe_name(raw)
        if not name:
            print("Nombre inválido.")
            return None

        if name in classes:
            cid = classes[name]
            print(f"⚠️ Esa clase ya existe con id={cid}.")
            return cid

        cid = len(classes)
        classes[name] = cid
        counters["per_class"][str(cid)] = 0

        tpl_gray = capture_template(cap, name)
        if tpl_gray is None:
            print("❌ No se capturó template. Clase no agregada.")
            classes.pop(name, None)
            counters["per_class"].pop(str(cid), None)
            return None

        templates[cid] = tpl_gray

        tpl_path = os.path.join(project_dir, "templates", f"class_{cid:02d}_{name}.png")
        cv2.imwrite(tpl_path, tpl_gray)

        print(f"✅ Clase agregada: '{name}' con id={cid}. Template: {tpl_path}")
        return cid

    cap = cv2.VideoCapture(CFG["CAM_INDEX"])
    if not cap.isOpened():
        raise RuntimeError("No se pudo abrir la cámara. Revisa CAM_INDEX o permisos.")

    # --- Crear primera clase (obligatoria) ---
    print("\nVamos a crear la PRIMERA clase.\n")
    current_class_id = add_class_flow(cap)
    if current_class_id is None:
        cap.release()
        return

    meta = {
        "project": project,
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "train_split": CFG["TRAIN_SPLIT"],
        "threshold": CFG["THRESHOLD"],
        "roi_size": CFG["ROI_SIZE"],
        "classes": classes,
        "counters": counters
    }
    save_project_meta(project_dir, meta)
    write_data_yaml(project_dir, classes)

    print("\n=== CONTROLES ===")
    print("  [k] Guardar imagen + label (solo si detecta template de la clase activa)")
    print("  [b] Guardar BACKGROUND (imagen sin etiqueta)")
    print("  [n] Agregar nueva clase (pide nombre con ventana + captura template)")
    print("  [0-9] Cambiar clase activa (si existe ese id)")
    print("  [t] Ajustar threshold (sube 0.03, y si llega a 0.85 baja a 0.75)")
    print("  [q] Salir\n")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("❌ No se pudo leer frame.")
            break

        h, w = frame.shape[:2]
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        active_tpl = templates.get(current_class_id)
        det = multiscale_match(gray, active_tpl, threshold=CFG["THRESHOLD"]) if active_tpl is not None else None

        vis = frame.copy()
        active_name = get_class_name(classes, current_class_id)

        cv2.putText(vis, f"Proyecto: {project}", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        cv2.putText(vis, f"Clase activa: [{current_class_id}] {active_name}", (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        cv2.putText(vis, f"Threshold: {CFG['THRESHOLD']:.2f}", (10, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        if det:
            cv2.rectangle(vis, (det.bx, det.by), (det.bx + det.bw, det.by + det.bh), (0, 255, 0), 2)
            if CFG["SHOW_DEBUG"]:
                cv2.putText(vis, f"OK score={det.score:.2f}", (det.bx, max(det.by - 10, 20)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
        else:
            cv2.putText(vis, "SIN DETECCION (mueva objeto / ajuste threshold)", (10, h - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 255), 2)

        cv2.imshow("YOLO Dataset Builder", vis)
        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):
            break

        # Guardar etiquetado
        if key == ord('k'):
            if det is None:
                print("⚠️ No guardado: no hay detección.")
                continue

            subset = "train" if random.random() < CFG["TRAIN_SPLIT"] else "val"
            classes_rev = {v: k for k, v in classes.items()}
            cname = classes_rev[current_class_id]

            counters["images_total"] += 1
            counters["per_class"][str(current_class_id)] = counters["per_class"].get(str(current_class_id), 0) + 1
            idx = counters["per_class"][str(current_class_id)]

            base = f"{cname}_{idx:05d}"
            img_path = os.path.join(project_dir, "images", subset, f"{base}.jpg")
            lbl_path = os.path.join(project_dir, "labels", subset, f"{base}.txt")

            cv2.imwrite(img_path, frame, [int(cv2.IMWRITE_JPEG_QUALITY), CFG["SAVE_JPEG_QUALITY"]])

            line = yolo_label_line(current_class_id, det.bx, det.by, det.bw, det.bh, w, h)
            with open(lbl_path, "w", encoding="utf-8") as f:
                f.write(line + "\n")

            print(f"📸 Guardado: {base} -> {subset} | {line}")

            if idx % CFG["AUTO_SAVE_EVERY"] == 0:
                meta["classes"] = classes
                meta["updated_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
                meta["counters"] = counters
                save_project_meta(project_dir, meta)
                write_data_yaml(project_dir, classes)

        # Guardar background
        if key == ord('b'):
            subset = "train" if random.random() < CFG["TRAIN_SPLIT"] else "val"
            counters["background_total"] += 1
            base = f"bg_{counters['background_total']:05d}"
            img_path = os.path.join(project_dir, "images", "background", subset, f"{base}.jpg")
            cv2.imwrite(img_path, frame, [int(cv2.IMWRITE_JPEG_QUALITY), CFG["SAVE_JPEG_QUALITY"]])
            print(f"🧱 Background guardado: {base} -> background/{subset}")

        # Agregar nueva clase (sin input bloqueante)
        if key == ord('n'):
            cid_new = add_class_flow(cap)
            if cid_new is not None:
                current_class_id = cid_new
                meta["classes"] = classes
                meta["updated_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
                meta["counters"] = counters
                save_project_meta(project_dir, meta)
                write_data_yaml(project_dir, classes)

        # Cambiar clase con dígitos
        if key >= ord('0') and key <= ord('9'):
            cid = int(chr(key))
            if cid in templates:
                current_class_id = cid
                print(f"➡️ Clase activa ahora: [{cid}] {get_class_name(classes, cid)}")
            else:
                print(f"⚠️ No existe template para class_id={cid}. (Agrega con 'n')")

        # Ajustar threshold
        if key == ord('t'):
            if CFG["THRESHOLD"] >= 0.85:
                CFG["THRESHOLD"] = 0.75
            else:
                CFG["THRESHOLD"] += 0.03
            print(f"🎚️ Threshold ahora: {CFG['THRESHOLD']:.2f}")

    cap.release()
    cv2.destroyAllWindows()

    meta["classes"] = classes
    meta["finished_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
    meta["counters"] = counters
    save_project_meta(project_dir, meta)
    write_data_yaml(project_dir, classes)

    print("\n✅ Proceso terminado.")
    print(f"Dataset creado en: {project_dir}/")
    print(f"- YOLO yaml: {project_dir}/data.yaml")
    print(f"- Templates: {project_dir}/templates/")
    print(f"- Images: {project_dir}/images/train y images/val (y background/...)")
    print(f"- Labels: {project_dir}/labels/train y labels/val")

# Ejecuta
main()


=== GENERADOR DE DATASET (YOLO) - JUPYTER FRIENDLY ===

Vamos a crear la PRIMERA clase.


--- Captura de TEMPLATE para: Topiramato ---
Coloca el objeto centrado y presiona 'c' para capturar (q para cancelar).
Tip: da CLICK en la ventana 'Captura Template' antes de presionar teclas.
✅ Clase agregada: 'Topiramato' con id=0. Template: Medicina\templates\class_00_Topiramato.png

=== CONTROLES ===
  [k] Guardar imagen + label (solo si detecta template de la clase activa)
  [b] Guardar BACKGROUND (imagen sin etiqueta)
  [n] Agregar nueva clase (pide nombre con ventana + captura template)
  [0-9] Cambiar clase activa (si existe ese id)
  [t] Ajustar threshold (sube 0.03, y si llega a 0.85 baja a 0.75)
  [q] Salir


--- Captura de TEMPLATE para: Enalapril ---
Coloca el objeto centrado y presiona 'c' para capturar (q para cancelar).
Tip: da CLICK en la ventana 'Captura Template' antes de presionar teclas.
✅ Clase agregada: 'Enalapril' con id=1. Template: Medicina\templates\class_01_Enalapril.pn